# Prédire le QI à 3 ans à partir de la HRV néonatale

*Groupe A · Jade · PRI 2026 · cohorte AuBE (CHU Saint-Étienne)*

**Question** : les indices de variabilité cardiaque (HRV) enregistrés à la naissance aident-ils à prédire le QI mesuré vers 3 ans (WPPSI-III), au-delà des facteurs cliniques ? Dans le rapport de Samuel Fornet, aucun modèle ne faisait mieux que prédire le QI moyen (MAE ≈ 10,3, R² ≤ 0).

| Plan de la semaine | Section |
|---|---|
| **Mardi** · un enregistrement par enfant, scores complets, fusion, nettoyage | 1 |
| **Mardi** · réduction des variables : corrélation > 0,95, variance quasi nulle, forêt aléatoire, PCA | 2 |
| **Mercredi** · Elastic Net, Random Forest, LightGBM, XGBoost, SVR | 3 |
| **Jeudi** · 10 × 5 validation croisée, QIT / QIV / QIP / NCL, MAE, MSE, RMSE, R², `model_comparison.csv` | 4 |
| **Pourquoi ça n'apporte rien**, et à partir de quand ça aurait apporté quelque chose | 4.4 |
| **Borne** : quelle part du QI la HRV peut-elle expliquer, au plus ? | 4.5 |
| **Contrôle positif** : la HRV retrouve-t-elle la prématurité ? | 5 |
| **Classification** QIT < 90 | 6 |
| **Hypothèses** : HRV plus tardive (M6 à M24) et trajectoires | 7 |
| **Conseils du prof** · prédiction conforme, explication des variables | 8, 9 |
| **Vendredi** · résumé pour le tuteur (`SUMMARY.md`), vérification des données | 10, 11 |

**Organisation** : ce notebook montre les étapes et les résultats. Le code technique (lecture, nettoyage, modèles, graphiques) est dans `outils_qit.py`, à côté, rangé selon les mêmes sections.

**Lancer** : `RAPIDE = True` pour vérifier que tout tourne (1 répétition, quelques minutes), puis `RAPIDE = False` pour les résultats à rapporter (10 répétitions, environ 25 minutes).

> **Données de santé d'enfants.** Les données et tout ce qui en est tiré au niveau individuel restent dans `data/` (ignoré par git). Le filtre `nbstripout` vide les sorties du notebook à chaque commit. Ne jamais donner ces données à un outil d'IA ; les sections 4.3 et 8 montrent des enfants un par un.

In [ ]:
import re
import subprocess
import sys
import time
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

# Racine du dépôt : le premier dossier qui contient evaluate_model.py, en remontant depuis celui du notebook.
RACINE = next((d for d in [Path.cwd(), *Path.cwd().parents] if (d / "evaluate_model.py").exists()), None)
if RACINE is None:
    raise FileNotFoundError("evaluate_model.py introuvable : ouvrir le notebook depuis le dépôt FSIQ-from-ECG.")
sys.path.insert(0, str(RACINE))
from evaluate_model import evaluate, load_labels   # évaluation commune de l'équipe
import outils_qit as outils                         # le code technique de ce notebook

In [ ]:
RAPIDE = False                          # True : vérification (1 répétition) ; False : résultats à rapporter (10)
CIBLES = ["QIT", "QIV", "QIP", "NCL"]  # QIT = cible principale
T_REPETITIONS, K_PLIS = 10, 5          # 10 × 5 plis pour évaluer (plan du jeudi)
N_ITER = 4 if RAPIDE else 12           # combinaisons essayées par RandomizedSearchCV (RF, LightGBM, XGBoost, SVR)
INDICES_A_PRIORI = ["RMSSD (TD24h)", "HF (FT24h)", "SDNN (TD24h)"]   # section 4.5 : choisis AVANT de voir les résultats
SEUIL_QIT = 90                         # section 6 : QIT < 90 contre ≥ 90
PASSAGES = ["M0", "M6", "M12", "M18", "M24"]    # section 7, H1 : HRV mesurée plus tard
TRAJECTOIRES = [("M0", "M12"), ("M0", "M24")]   # section 7, H2 : évolution de la HRV entre deux âges
ALPHA = 0.10                           # section 8 : intervalles à 1 − α = 90 %
GRAINE = 42
# Seuils de nettoyage et de sélection (corrélation 0,95, top 30, PCA 90 %…) : en tête de outils_qit.py.

REPETITIONS = [1] if RAPIDE else list(range(1, T_REPETITIONS + 1))
DOSSIER_DONNEES = RACINE / "data"                 # données : local seulement
DOSSIER_PARTAGE = DOSSIER_DONNEES / "cleaned"     # cohorte, plis, prédictions : local seulement
DOSSIER_RESULTATS = Path.cwd() / "results"        # chiffres agrégés : model_comparison.csv, SUMMARY.md, graphique
FICHIERS = {"lin": "HRV_Lineaire_Full.csv", "nl": "HRV_NonLineaire_Full.csv", "wppsi": "Résultats WPPSI III.csv"}

# 1. Preprocessing

## 1.1 Lecture des fichiers

Trois exports Excel. Les fichiers HRV ont deux lignes d'en-tête : les noms, puis une ligne d'unités (M/F, 0/1, ms, %…) qui est ignorée. Le fichier WPPSI a aussi deux lignes d'en-tête, réunies en « Bloc | Colonne ». Les cases vides peuvent être écrites `""`, `" "`, `-` ou `NaN` : toutes deviennent des valeurs manquantes.

Les fichiers sont reliés par le **numéro** du sujet (1re colonne), harmonisé comme dans `evaluate_model.py` (« 012 », « 12.0 » → « 12 »). Des initiales se répètent d'un enfant à l'autre (voir `data/pulisci_righe.py`) ; la date de naissance sert donc de vérification (1.2).

| Score WPPSI | Ce qu'il mesure |
|---|---|
| **QIT** | quotient intellectuel total (cible principale) |
| **QIV** | quotient verbal : langage, connaissances |
| **QIP** | quotient de performance : raisonnement non verbal, visuo-spatial |
| **NCL** | note composite de langage : vocabulaire |

Moyenne 100, écart-type 15 dans la population. Ces scores ne servent jamais de variables d'entrée. Un enfant sans tous ses scores est retiré (perdu de vue).

In [ ]:
sources = outils.charger(DOSSIER_DONNEES, FICHIERS)
CIBLES = [c for c in CIBLES if c in sources.scores.columns]
display(sources.scores.describe().loc[["mean", "std"]].round(1))

## 1.2 Un jeu de données par passage

Pour un passage donné (M0 = naissance, puis M6, M12, M18, M24), on garde un enregistrement par enfant, les indices HRV des deux fichiers, les covariables cliniques et les scores. On retire ensuite :
- les enfants absents d'un des 3 fichiers, ou dont la date de naissance diffère d'un fichier à l'autre ;
- les colonnes vides à ce passage : à M0, le fichier linéaire n'a que ses indices **24h** ;
- les variables manquantes pour plus de 30 % des enfants, ou quasi constantes (une même valeur pour 95 % des enfants) ;
- les covariables cliniques trop rares (moins de 5 enfants dans une classe). Le sexe est codé 0/1.

Aucune de ces étapes ne regarde le QI : rien n'apprend sur la cible, on peut donc les faire avant de découper les plis. Le détail est affiché pour M0, l'analyse principale ; les autres passages servent aux hypothèses de la section 7.

In [ ]:
donnees = outils.construire_donnees(sources, "M0", bavard=True)

## 1.3 Cohorte et plis communs

`evaluate_model.py` ne compare des modèles que s'ils ont prédit **les mêmes enfants avec les mêmes plis** : `cohort.csv` (enfants et scores) et `folds.csv` (10 répétitions de 5 plis stratifiés sur le QIT). Le notebook les écrit à chaque lancement dans `data/cleaned/` : mêmes données et même graine, donc mêmes plis. Si l'équipe fournit un jour les siens (`prepare_data.py`), il suffira de retirer la ligne `creer_plis`.

In [ ]:
fichier_cohorte, fichier_plis = DOSSIER_PARTAGE / "cohort.csv", DOSSIER_PARTAGE / "folds.csv"
outils.creer_plis(donnees, fichier_cohorte, fichier_plis, T_REPETITIONS, K_PLIS, GRAINE)
donnees = donnees.sort_index(key=lambda ids: ids.astype(int))      # enfants triés par numéro
donnees.rename_axis("child_id").to_csv(DOSSIER_PARTAGE / "dataset_m0.csv")
etiquettes = load_labels(fichier_cohorte, CIBLES).loc[donnees.index]
plis = outils.plis_communs(fichier_plis, REPETITIONS)     # (répétition, pli, enfants d'entraînement, enfants de test)

X = donnees.drop(columns=list(sources.scores.columns))
cols_lin, cols_nl, cols_clin = ([c for c in X.columns if c.startswith(p)] for p in ("lin|", "nl|", "clin|"))
print(f"Jeu final : {len(X)} enfants ; {len(cols_lin)} indices linéaires, {len(cols_nl)} non linéaires, "
      f"{len(cols_clin)} covariables cliniques ({', '.join(c[5:] for c in cols_clin)}) ; "
      f"{len(REPETITIONS)} × {K_PLIS} plis")

## 1.4 Y a-t-il un signal dans les variables ?

Avant tout modèle, un repère simple : la corrélation de chaque variable avec le QIT. Avec beaucoup de variables, certaines sont corrélées au QIT **par pur hasard** ; on compare donc la plus forte corrélation observée à celle obtenue en mélangeant les QIT entre enfants. Si les deux sont proches, aucune variable n'est liée au QIT, même prise seule, et aucun modèle ne pourra faire de miracle.

C'est descriptif : choisir des variables à partir de ces corrélations, hors des plis, serait une fuite d'information.

In [ ]:
rho, rho_hasard, seuil_hasard = outils.correlation_contre_hasard(X, etiquettes["QIT"])
SIGNAL_UNIVARIE = rho.iloc[0] > seuil_hasard
print(f"Plus forte corrélation |Spearman| avec le QIT : {rho.iloc[0]:.2f}. Par pur hasard, avec {X.shape[1]} variables, "
      f"on obtient en moyenne {rho_hasard:.2f}, et jusqu'à {seuil_hasard:.2f} une fois sur 20.")
print("Conclusion :", "une corrélation dépasse le hasard, un lien faible existe peut-être." if SIGNAL_UNIVARIE else
      "rien ne dépasse le hasard : aucune variable n'est liée au QIT, même prise seule.")
print("Les 5 plus fortes :", rho.head(5).round(2).to_dict())

# 2. Feature selection

Avec environ 70 variables HRV pour 189 enfants, un modèle peut apprendre le bruit des enfants d'entraînement au lieu d'un vrai lien. On réduit donc les variables **dans chaque pli, sur les seuls enfants d'entraînement** (sinon les enfants de test « aident » à préparer le modèle et les résultats sont trop optimistes) :

1. **Toujours** : remplir les trous par la médiane, garder une seule variable par groupe corrélé à plus de 0,95, transformer chaque indice HRV vers une loi normale (ils sont très asymétriques).
2. **Au choix, selon l'expérience** :

| Option | Principe |
|---|---|
| `toutes` | pas de sélection : le modèle fait le tri lui-même (Elastic Net met des coefficients à zéro) |
| `sél. RF` | les 30 variables les plus importantes d'une forêt aléatoire de référence (conseil du prof) |
| `PCA 90 %` | les composantes principales qui gardent 90 % de la variance (conseil du prof) |

Trois jeux de variables : `Clinique seul` (la référence clinique : ce que la HRV devrait améliorer), `HRV seule`, `Clinique + HRV`. Ci-dessous, un exemple de pipeline complet : tout ce bloc est ajusté dans chaque pli (cliquer sur une étape pour voir ses réglages).

In [ ]:
JEUX = {"Clinique seul": cols_clin, "HRV seule": cols_lin + cols_nl, "Clinique + HRV": cols_clin + cols_lin + cols_nl}
outils.faire_pipeline(JEUX["Clinique + HRV"], "LightGBM", "sél. RF", N_ITER)

# 3. Models

Cinq modèles de régression, réglés dans une **boucle interne à 5 plis** sur les seuls enfants d'entraînement :
- **Elastic Net** : régression linéaire pénalisée, la référence ; `ElasticNetCV` choisit la force de la pénalité ;
- **Random Forest, LightGBM, XGBoost** : arbres de décision, capables de liens non linéaires (le prof conseille LightGBM et XGBoost) ;
- **SVR** : régression à noyau.

Pour ces quatre derniers, `RandomizedSearchCV` essaie `N_ITER` combinaisons tirées dans les espaces d'hyperparamètres du plan du mercredi (`outils.ESPACES`). Une **expérience** = un modèle + une sélection + un jeu de variables ; toutes voient les mêmes enfants et les mêmes plis.

Un 6e modèle sert aux sections 5 et 6 : la **régression logistique**, pour prédire une classe (prématuré ou non, QIT < 90 ou non).

In [ ]:
EXPERIENCES = [                               # (modèle, sélection, jeu de variables)
    ("Elastic Net", "toutes", "Clinique seul"),
    ("Elastic Net", "toutes", "Clinique + HRV"),
    ("Random Forest", "toutes", "Clinique + HRV"),
    ("LightGBM", "sél. RF", "Clinique + HRV"),
    ("XGBoost", "sél. RF", "Clinique + HRV"),
    ("SVR", "PCA 90 %", "Clinique + HRV"),
]
EXPERIENCES = [e for e in EXPERIENCES if JEUX[e[2]]]

# 4. Evaluation

Protocole du jeudi : **10 répétitions × 5 plis** (`folds.csv`) ; dans chaque pli, le modèle apprend sur 4/5 des enfants et prédit le 5e. Les prédictions sont écrites au format de `evaluate_model.py` (`data/cleaned/predictions/`), donc comparables avec le reste de l'équipe. Avec `RAPIDE = True`, seule la 1re répétition est faite.

## 4.1 Lancement

In [ ]:
DOSSIER_PRED = DOSSIER_PARTAGE / "predictions"
DOSSIER_PRED.mkdir(parents=True, exist_ok=True)
predictions, fichiers = {}, {}
debut = time.time()
for modele, selection, jeu in EXPERIENCES:
    t, nom = time.time(), f"{modele} · {selection} · {jeu}"
    predictions[nom] = outils.predictions_croisees(X[JEUX[jeu]], etiquettes, plis, modele, selection, N_ITER)
    fichiers[nom] = DOSSIER_PRED / f"predictions_{re.sub(r'[^a-z0-9]+', '_', outils.normaliser_texte(nom)).strip('_')}.csv"
    outils.ecrire_predictions(predictions[nom], plis, fichier_plis, fichier_cohorte, fichiers[nom], REPETITIONS)
    print(f"{nom} : {time.time() - t:.0f} s")
print(f"Total : {(time.time() - debut) / 60:.1f} min")

## 4.2 Comparaison des modèles

Une mesure par pli de test, résumée par moyenne ± écart-type (MAE, MSE, RMSE, R²) et écrite dans `results/model_comparison.csv`. La **référence** prédit, dans chaque pli, le score moyen des enfants d'entraînement : c'est le plancher à battre. Le **gain** est ce que le modèle gagne en MAE sur cette référence ; son intervalle de confiance à 95 % vient de `evaluate_model.py` (bootstrap sur les enfants). **Si l'intervalle du gain contient 0, l'amélioration n'est pas démontrée.**

In [ ]:
par_pli = outils.mesures_par_pli(predictions, etiquettes, plis)
comparaison_modeles = outils.comparaison_par_modele(par_pli)
DOSSIER_RESULTATS.mkdir(parents=True, exist_ok=True)
comparaison_modeles.round(4).to_csv(DOSSIER_RESULTATS / "model_comparison.csv", index=False)
resume_equipe, _ = evaluate(fichier_plis, fichier_cohorte, [str(f) for f in fichiers.values()],
                            names=list(fichiers), targets=CIBLES, verbose=False)
display(outils.tableau_resultats(comparaison_modeles, resume_equipe, CIBLES))
outils.graphique_gains(par_pli, CIBLES, DOSSIER_RESULTATS / "comparaison_modeles.png")
modeles_seuls = par_pli[par_pli.model != outils.REFERENCE]
MEILLEURS = {c: modeles_seuls[modeles_seuls.target == c].groupby("model")["MAE"].mean().idxmin() for c in CIBLES}

## 4.3 Prédit contre observé (QIT, meilleur modèle)

Un point = un enfant, prédit quand il était dans le pli de test (1re répétition). **Graphique individuel : à ne pas partager.** Si les points forment une bande horizontale autour du QIT moyen, le modèle prédit presque la même chose pour tout le monde : il n'a pas trouvé de signal.

In [ ]:
MEILLEUR = MEILLEURS["QIT"]
modele_m, selection_m, jeu_m = next(e for e in EXPERIENCES if " · ".join(e) == MEILLEUR)
vrai, pred = etiquettes["QIT"], predictions[MEILLEUR]["QIT"].loc[REPETITIONS[0]]
outils.graphique_predit_observe(vrai, pred, MEILLEUR)
erreur = (vrai - pred).abs()
print(f"Écart-type des prédictions : {pred.std():.1f} points (celui des vrais QIT : {vrai.std():.1f}). "
      f"MAE pour les QIT < {SEUIL_QIT} : {erreur[vrai < SEUIL_QIT].mean():.1f} points, "
      f"pour les autres : {erreur[vrai >= SEUIL_QIT].mean():.1f}.")

## 4.4 Pourquoi ça n'apporte rien, et quand ça aurait apporté quelque chose

**Pourquoi.** Trois raisons, de la plus importante à la moins importante :
1. **Il n'y a pas de lien dans les variables** (section 1.4) : aucune variable n'est plus corrélée au QIT que ce que donne le hasard. Un modèle ne crée pas un signal absent ; il ne peut que combiner ceux qui existent.
2. **Avec 189 enfants, chercher un signal qui n'existe pas coûte cher.** Chaque coefficient appris sur du bruit ajoute de l'erreur. Elastic Net le sent et met presque tout à zéro (il retombe sur la moyenne) ; les arbres, plus flexibles, apprennent le bruit et font **moins bien** que la moyenne (section 4.2).
3. **Le QI à 3 ans dépend de beaucoup de choses absentes ici** : l'environnement familial, la stimulation, le niveau socio-économique, les événements des trois premières années. La HRV à la naissance ne voit rien de tout cela.

**Quand ça aurait apporté quelque chose.** Pour le savoir, on fabrique un **faux QI** qui dépend vraiment de 5 indices HRV, avec une force connue : il explique 0 %, 5 %, 10 %, 20 % ou 30 % de la variance du QI (le R² « planté »), le reste étant du bruit. On lance ensuite exactement la même méthode (Elastic Net, Clinique + HRV, mêmes enfants). On voit ainsi à partir de quelle force de lien notre méthode l'aurait détecté, et on situe notre vrai résultat sur cette échelle. La section 4.5 borne ensuite ce lien avec un test plus sensible.

In [ ]:
gardees, total = outils.variables_gardees(X[JEUX["Clinique + HRV"]], etiquettes["QIT"], plis), len(JEUX["Clinique + HRV"])
print(f"Elastic Net (Clinique + HRV, QIT) garde en moyenne {gardees:.1f} variables sur {total} par pli"
      + (" : presque aucune, il retombe sur la moyenne faute de lien fiable." if gardees < 0.1 * total else "."))

simulation = outils.simulation_detection(X, JEUX["Clinique + HRV"], cols_lin + cols_nl, etiquettes["QIT"],
                                         [0.0, 0.05, 0.10, 0.20, 0.30], len(REPETITIONS), K_PLIS, GRAINE)
display(simulation.round(2))
nom_en = "Elastic Net · toutes · Clinique + HRV"
reel = resume_equipe.set_index(["target", "model"]).loc[("QIT", nom_en)] if nom_en in fichiers else None
outils.graphique_simulation(simulation, reel)

detectes = simulation.index[simulation["IC bas"] > 0]
SEUIL_DETECTION = detectes.min() if len(detectes) else None
print((f"Avec ces {len(X)} enfants, la méthode détecte un lien dès qu'il explique environ "
       f"{SEUIL_DETECTION:.0%} de la variance du QI." if SEUIL_DETECTION is not None else
       "Aucun niveau simulé n'est détecté avec cette répétition : relancer avec RAPIDE = False.")
      + (" Notre vrai gain n'est pas démontré : si la HRV néonatale explique une part du QI à 3 ans, "
         "elle est plus petite que ce seuil." if reel is not None and reel.gain_ci_low <= 0 else ""))

## 4.5 Quelle part du QI la HRV peut-elle expliquer, au plus ?

D'après la section 4.4, la prédiction ne voit un lien qu'à partir d'environ 20 % de variance expliquée. « On n'a rien vu » laisse donc ouverte la possibilité d'un lien plus faible, de 5 ou 10 %. Pour le borner, on change de question : au lieu de **prédire** le QI avec une centaine de variables (et de payer le prix de chercher parmi toutes), on **teste** quelques indices **choisis à l'avance**. Avec 4 tests ciblés, la méthode est beaucoup plus sensible.

Les indices sont choisis d'après la littérature, sans regarder nos résultats (`INDICES_A_PRIORI`) :
- **RMSSD** et **HF** (24h) : marqueurs du **tonus vagal**, le frein que le système parasympathique exerce sur le cœur. La théorie polyvagale de Porges relie le tonus vagal du nouveau-né à son développement ultérieur (par exemple Doussard-Roosevelt et al., 1997, chez des prématurés) ;
- **SDNN** (24h) : variabilité globale ;
- la **1re composante principale** de tous les indices HRV : un résumé de l'ensemble, calculé sans regarder le QI.

Pour chacun, on calcule la corrélation de Spearman avec le QIT, brute puis **ajustée** sur les covariables cliniques (ce que la HRV apporterait au-delà de la prématurité, du sexe…), avec son intervalle de confiance (bootstrap sur les enfants). Les intervalles sont élargis pour tenir compte des 4 tests (correction de Bonferroni) : les 4 bornes tiennent **ensemble** avec 95 % de confiance.

**Lecture** :
- si un intervalle exclut 0, le lien est démontré, même s'il est faible ;
- sinon, la borne haute au carré donne la **plus grande part de la variance du QIT compatible avec nos données** (variance des rangs, ce qui revient presque au même). « La HRV explique au plus X % » est un résultat, plus fort que « on n'a rien vu ».

Les corrélations brutes et ajustées forment deux familles de 4 tests, chacune à 95 % de confiance. La question principale est la corrélation brute ; l'ajustée la complète. Un lien brut qui disparaît après ajustement passe sans doute par les covariables cliniques (la prématurité change à la fois la HRV et le QI), pas par la HRV elle-même.

*Vérifié sur 8 000 jeux simulés de 189 enfants dont on connaît la vérité (`python verifier_bornes.py`, environ 1 minute) :*
- *sans aucun lien, au plus 4 % de faux positifs ;*
- *un lien au seuil annoncé est détecté 77 fois sur 100 ;*
- *un faux lien créé par la prématurité est éliminé par l'ajustement (4 % de faux positifs ajustés, contre 32 % en brut).*

> Ne pas changer `INDICES_A_PRIORI` après avoir vu les résultats : choisir les indices qui marchent le mieux rendrait le test invalide (c'est la même fuite d'information qu'en section 1.4).

In [ ]:
cols_hrv = cols_lin + cols_nl
choisis = {nom: outils.trouver_colonne(X[cols_hrv], nom) for nom in INDICES_A_PRIORI}
if None in choisis.values():
    print("Indices introuvables, ignorés :", [n for n, c in choisis.items() if c is None])
indices = X[[c for c in choisis.values() if c is not None]].rename(columns=lambda c: c.split("|", 1)[1])
indices["1re composante HRV"], part_cp1 = outils.premiere_composante(X, cols_hrv)
print(f"La 1re composante principale résume {part_cp1:.0%} de la variance des {len(cols_hrv)} indices HRV.")

bornes = outils.bornes_correlation(indices, etiquettes["QIT"], X[cols_clin])
VARIANCE_DETECTABLE = outils.variance_detectable(len(X), indices.shape[1])
display(outils.tableau_bornes(bornes))
outils.graphique_bornes(bornes, np.sqrt(VARIANCE_DETECTABLE))

BORNE, BORNE_AJUSTEE = bornes["variance max"].max(), bornes["variance max ajusté"].max()
LIENS_DEMONTRES = ([f"{i} (brut)" for i in bornes.index[bornes["démontré"] == "oui"]]
                   + [f"{i} (ajusté)" for i in bornes.index[bornes["démontré ajusté"] == "oui"]])
print(f"Sensibilité : avec {len(X)} enfants et {indices.shape[1]} tests, un lien qui explique {VARIANCE_DETECTABLE:.0%} "
      "de la variance du QIT est détecté 8 fois sur 10"
      + (f" (contre {SEUIL_DETECTION:.0%} pour la prédiction, section 4.4)." if SEUIL_DETECTION is not None else "."))
BRUT_SEULEMENT = list(bornes.index[(bornes["démontré"] == "oui") & (bornes["démontré ajusté"] == "non")])
print((f"Lien démontré pour : {', '.join(LIENS_DEMONTRES)}. " if LIENS_DEMONTRES else "Aucun lien démontré. ")
      + f"Avec 95 % de confiance, aucun de ces indices n'explique plus de {BORNE:.0%} de la variance du QIT "
      f"({BORNE_AJUSTEE:.0%} au-delà des covariables cliniques).")
if BRUT_SEULEMENT:
    print(f"Attention : le lien de {', '.join(BRUT_SEULEMENT)} disparaît après ajustement sur les covariables cliniques. "
          "Il passe sans doute par elles (par exemple la prématurité, qui change à la fois la HRV et le QI), "
          "pas par la HRV elle-même.")

# 5. Contrôle positif : la HRV retrouve-t-elle la prématurité ?

Un résultat nul peut avoir deux causes très différentes : **il n'y a pas de lien**, ou **notre méthode (ou nos données) est incapable d'en voir un**. Le contrôle positif sert à les distinguer : on pose à la même méthode une question dont on **connaît** la réponse.

On sait que la HRV d'un nouveau-né dépend fortement de sa maturité : un enfant né prématurément a un système nerveux autonome moins mûr, donc une HRV différente. On demande donc aux seuls indices HRV de deviner si l'enfant est prématuré.

- Si la HRV retrouve bien la prématurité (AUC nettement au-dessus de 0,5), **les données HRV contiennent de l'information physiologique et la méthode sait l'extraire**. L'absence de lien avec le QI devient alors un vrai résultat, pas un problème de données ou de code.
- Si elle n'y arrive pas, il faut d'abord se méfier des données (qualité des enregistrements, définition de la prématurité dans le fichier).

L'**AUC** se lit ainsi : 0,5 = le modèle ne fait pas mieux qu'une pièce de monnaie ; 1 = il sépare parfaitement les deux groupes ; au-dessus de 0,7, on parle en général d'une bonne discrimination.

In [ ]:
controle = None
col_prema = "clin|Prématuré"
if col_prema not in X or X[col_prema].nunique() != 2:
    print("Prématurité indisponible : pas de contrôle positif.")
else:
    renseigne = X[col_prema].dropna()
    prema = (renseigne == renseigne.max()).astype(int)          # 1 = prématuré (codage 0/1 du fichier)
    print(f"{prema.sum()} enfants prématurés sur {len(prema)} renseignés")
    if min(prema.sum(), (1 - prema).sum()) < 10:
        print("Moins de 10 enfants dans un des deux groupes : contrôle trop fragile pour conclure.")
    else:
        plis_prema = outils.plis_stratifies(prema, len(REPETITIONS), K_PLIS, GRAINE)
        P_prema = outils.predictions_croisees(X.loc[prema.index, cols_lin + cols_nl], prema, plis_prema, "Logistique")
        controle = outils.resume_classification(prema, P_prema)
        display(pd.DataFrame([controle], index=["HRV seule → prématurité"]).round(2))
        print("Verdict :", "la HRV retrouve bien la prématurité : les données et la méthode sont capables de voir un lien."
              if controle["AUC IC bas"] > 0.5 else
              "la HRV ne retrouve pas la prématurité de façon démontrée : se méfier d'abord des données.")

# 6. Classification : repérer les enfants avec un QIT < 90

Prédire le QI au point près est peut-être trop demander. Une question plus modeste, et plus utile en clinique : **peut-on repérer les enfants dont le QIT sera inférieur à 90** (environ 1 enfant sur 5), pour les suivre de plus près ? Un signal limité aux QI bas pourrait passer inaperçu dans la régression.

**Le piège du déséquilibre.** Avec environ 80 % d'enfants au-dessus de 90, un « modèle » qui répond toujours « ≥ 90 » a 80 % d'exactitude, sans rien savoir. L'exactitude seule est donc trompeuse. On regarde plutôt :
- l'**AUC** : 0,5 = hasard, 1 = parfait ;
- la **sensibilité** (part des enfants < 90 repérés) et la **spécificité** (part des enfants ≥ 90 correctement laissés de côté) ;
- l'**exactitude équilibrée** (moyenne des deux : 0,5 = hasard) ;
- le **kappa de Cohen** (accord au-delà du hasard : 0 = hasard, 1 = parfait).

Le modèle est une régression logistique pénalisée, les deux classes comptant autant (`class_weight="balanced"`). Il est évalué sur les mêmes plis que la régression. La ligne « toujours ≥ 90 » montre le piège de l'exactitude.

In [ ]:
bas = (etiquettes["QIT"] < SEUIL_QIT).astype(int)
print(f"{bas.sum()} enfants sur {len(bas)} ({bas.mean():.0%}) ont un QIT < {SEUIL_QIT}")
classification = {f"toujours « ≥ {SEUIL_QIT} »": {"AUC": 0.5, "AUC IC bas": 0.5, "AUC IC haut": 0.5,
                                                  "sensibilité": 0.0, "spécificité": 1.0,
                                                  "exactitude": 1 - bas.mean(), "exactitude équilibrée": 0.5, "kappa": 0.0}}
for jeu in ["Clinique seul", "Clinique + HRV"]:
    if JEUX[jeu]:
        P = outils.predictions_croisees(X[JEUX[jeu]], bas, plis, "Logistique")
        classification[f"Logistique · {jeu}"] = outils.resume_classification(bas, P)
classification = pd.DataFrame(classification).T
display(classification.round(2))

# 7. Hypothèses : HRV plus tardive et trajectoires

La HRV de la naissance ne prédit pas le QI à 3 ans. Deux hypothèses à tester avant de conclure que la HRV n'y est pour rien :

- **H1 : la HRV mesurée plus tard prédit mieux.** À 6, 12, 18 ou 24 mois, le système nerveux autonome est plus mûr et plus stable, et la mesure est plus proche de l'évaluation à 3 ans.
- **H2 : c'est l'évolution qui compte, pas la valeur à un âge donné.** Un enfant dont la HRV mûrit vite entre la naissance et 12 (ou 24) mois pourrait avoir un meilleur développement. Les variables sont alors les différences d'indices HRV entre les deux âges.

**Test** : pour chaque analyse, Elastic Net avec les covariables cliniques et les indices HRV, comparé au QIT moyen, en validation croisée sur les enfants qui ont les enregistrements nécessaires. Une hypothèse est soutenue si l'intervalle de confiance du gain est au-dessus de 0.

**Précautions** : les enfants diffèrent d'une analyse à l'autre (tous n'ont pas tous les enregistrements), donc les résultats ne se comparent pas au point près. Et avec 7 tests, un résultat positif isolé peut être dû au hasard : il faudrait le confirmer sur d'autres données avant d'y croire.

In [ ]:
lignes = []


def tester(d, analyse):
    """Elastic Net (toutes les variables de d sauf le QIT) contre le QIT moyen, en validation croisée."""
    colonnes = [c for c in d.columns if c.startswith(("lin|", "nl|", "evol|", "clin|"))]
    if len(d) < 50 or not colonnes:
        print(f"{analyse} : seulement {len(d)} enfants, analyse ignorée")
        return
    plis_h = outils.plis_stratifies(d["QIT"], len(REPETITIONS), K_PLIS, GRAINE)
    P = outils.predictions_croisees(d[colonnes], d["QIT"], plis_h)
    lignes.append({"analyse": analyse, "variables": len(colonnes), **outils.gain_vs_moyenne(d["QIT"], P, plis_h)})


jeux_passages = {p: outils.construire_donnees(sources, p) for p in PASSAGES}
for passage, d in jeux_passages.items():
    tester(d, f"H1 · HRV à {passage}")
for avant, apres in TRAJECTOIRES:
    d1, d2 = jeux_passages[avant], jeux_passages[apres]
    enfants = d1.index.intersection(d2.index)
    communes = [c for c in d1.columns if c.startswith(("lin|", "nl|")) and c in d2.columns]
    evolution = (d2.loc[enfants, communes] - d1.loc[enfants, communes]).add_prefix("evol|")
    tester(evolution.join(d1.loc[enfants, [c for c in d1.columns if c.startswith("clin|")] + ["QIT"]]),
           f"H2 · évolution {avant} → {apres}")

hypotheses = pd.DataFrame(lignes).set_index("analyse")
hypotheses["soutenue"] = np.where(hypotheses["IC bas"] > 0, "oui", "non")
display(hypotheses.round(2))
outils.graphique_hypotheses(hypotheses)

# 8. Prédiction conforme (QIT)

Un QIT prédit seul ne dit rien de sa marge d'erreur. La **prédiction conforme** donne un intervalle [prédiction − q ; prédiction + q] qui contient le vrai QIT avec une probabilité d'environ 90 %, quel que soit le modèle. La demi-largeur q se mesure sur les erreurs du modèle chez des enfants **qu'il n'a pas vus** : pour chaque pli de test, une validation croisée interne sur les seuls enfants d'entraînement. Les enfants de test ne servent jamais à calibrer leur propre intervalle, ce qui permet de vérifier la couverture honnêtement.

Si l'intervalle n'est pas plus étroit que celui construit autour du simple QIT moyen, le modèle n'apporte rien pour un enfant donné. **Graphique individuel : à ne pas partager.**

In [ ]:
r1 = REPETITIONS[0]
pred_r1 = predictions[MEILLEUR]["QIT"].loc[r1]
conforme = outils.intervalles_conformes(X[JEUX[jeu_m]], etiquettes["QIT"], pred_r1, [p for p in plis if p[0] == r1],
                                        ALPHA, modele_m, selection_m, N_ITER)
couverture = (np.abs(conforme.vrai - conforme.pred) <= conforme.q).mean()
couverture_ref = (np.abs(conforme.vrai - conforme.moyenne) <= conforme.q_ref).mean()
q_final = outils.quantile_conforme((etiquettes["QIT"] - pred_r1).abs().to_numpy(), ALPHA)
display(pd.DataFrame({"couverture observée": [f"{couverture:.0%}", f"{couverture_ref:.0%}"],
                      "demi-largeur (points de QIT)": [f"± {conforme.q.mean():.1f}", f"± {conforme.q_ref.mean():.1f}"]},
                     index=[MEILLEUR, "autour du QIT moyen"]))
print(f"Pour un nouvel enfant : QIT prédit ± {q_final:.1f} points, avec une probabilité d'environ {1 - ALPHA:.0%} "
      "de contenir son vrai QIT.")
outils.graphique_intervalles(conforme, SEUIL_QIT, ALPHA)

# 9. Explication des variables

L'importance d'une variable se mesure ainsi : on mélange ses valeurs entre les enfants de test et on regarde de combien la performance baisse. Si elle baisse beaucoup, le modèle s'appuyait sur cette variable.

- **Pour le QI**, aucun modèle ne fait mieux que la moyenne (section 4) : l'importance de ses variables ne mesurerait que du bruit, et Elastic Net met presque tous ses coefficients à zéro (section 4.4). La réponse honnête à « quelle variable compte pour le QI ? » est donc : **aucune, avec ces données**.
- **Pour la prématurité** (contrôle positif, section 5), la HRV a un vrai signal : c'est là que l'explication a un sens. Le graphique montre les indices HRV qui permettent de reconnaître un enfant prématuré.

Une variable importante est **associée** au résultat dans cette cohorte ; ce n'est pas une **cause**. « Que faire pour un enfant prédit sous 90 ? » est une question causale : un modèle prédictif peut au mieux aider à repérer des enfants à suivre, et les interventions relèvent d'études cliniques.

In [ ]:
if controle is None:
    print("Pas de contrôle positif exploitable : pas d'explication à montrer.")
else:
    importances = outils.importances_permutation(X.loc[prema.index, cols_lin + cols_nl], prema,
                                                 [p for p in plis_prema if p[0] == 1])
    outils.graphique_importances(importances, "Indices HRV qui reconnaissent la prématurité")

# 10. Résumé pour le tuteur

Généré à partir des résultats ci-dessus (chiffres agrégés seulement) et enregistré dans `results/SUMMARY.md`.

In [ ]:
equipe = resume_equipe.set_index(["target", "model"])
comp = comparaison_modeles.set_index(["target", "model"])
demontres = [c for c in CIBLES if equipe.loc[(c, MEILLEURS[c])].gain_ci_low > 0]
mae_qit = comp.xs("QIT")["mae_mean"]
seul, avec = "Elastic Net · toutes · Clinique seul", "Elastic Net · toutes · Clinique + HRV"
nom_classif = classification.iloc[1:]["AUC"].idxmax()
classif = classification.loc[nom_classif]
prema_retrouvee = controle is not None and controle["AUC IC bas"] > 0.5


def ligne_regression(cible):
    m, e = MEILLEURS[cible], equipe.loc[(cible, MEILLEURS[cible])]
    s, ref = comp.loc[(cible, m)], comp.loc[(cible, outils.REFERENCE)]
    return (f"| {cible} | {m} | {s.mae_mean:.2f} ± {s.mae_std:.2f} | {s.r2_mean:.3f} | {ref.mae_mean:.2f} | "
            f"{e.MAE_gain_vs_baseline:+.2f} {outils.ic(e.gain_ci_low, e.gain_ci_high)} "
            f"({'démontré' if e.gain_ci_low > 0 else 'non démontré'}) |")


SUMMARY = "\n".join([
    "# Prédiction du QI à 3 ans à partir de la HRV néonatale : résumé",
    f"*Groupe A · Jade · généré le {date.today():%d/%m/%Y} par `01_prediction_qit.ipynb` · "
    + ("mode rapide (1 répétition), chiffres provisoires*" if RAPIDE else f"{T_REPETITIONS} × {K_PLIS} plis*"),
    "",
    "## Conclusion",
    (f"**Gain démontré par rapport au score moyen pour : {', '.join(demontres)}.** " if demontres else
     "**Aucun modèle ne fait mieux que prédire le score moyen, sur aucune cible.** ")
    + (f"La HRV retrouve pourtant bien la prématurité (AUC {controle['AUC']:.2f}) : les données et la méthode savent "
       "voir un lien quand il existe." if prema_retrouvee else ""),
    "",
    "## Données et méthode",
    f"- {len(X)} enfants de la cohorte AuBE (enregistrement à la naissance, scores WPPSI-III complets) ; "
    f"{len(cols_lin)} indices HRV linéaires, {len(cols_nl)} non linéaires, {len(cols_clin)} covariables cliniques.",
    f"- Modèles : {', '.join(sorted({e[0] for e in EXPERIENCES}))} ; réduction des variables dans chaque pli ; "
    f"évaluation sur les plis communs de l'équipe ({len(REPETITIONS)} × {K_PLIS}, `evaluate_model.py`).",
    "",
    "## Régression (meilleur modèle par cible)",
    "| Cible | Modèle | MAE ± écart-type | R² | MAE de la moyenne | Gain vs moyenne [IC 95 %] |",
    "|---|---|---|---|---|---|",
    *[ligne_regression(c) for c in CIBLES],
    "",
    *([f"- Ajouter la HRV au modèle clinique (Elastic Net, QIT) : la MAE passe de {mae_qit[seul]:.2f} à "
       f"{mae_qit[avec]:.2f} ({'mieux' if mae_qit[avec] < mae_qit[seul] else 'moins bien'})."]
      if seul in mae_qit and avec in mae_qit else []),
    f"- Signal dans les variables : la plus forte corrélation avec le QIT vaut {rho.iloc[0]:.2f}, contre "
    f"{rho_hasard:.2f} en moyenne par hasard (jusqu'à {seuil_hasard:.2f})"
    + (" : une corrélation dépasse le hasard." if SIGNAL_UNIVARIE else " : rien ne dépasse le hasard, pas de lien à apprendre."),
    *([f"- Limite de détection (simulation) : avec ces enfants, la méthode voit un lien dès qu'il explique environ "
       f"{SEUIL_DETECTION:.0%} de la variance du QI" + ("." if demontres else " ; si la HRV en explique une part, elle est plus petite.")]
      if SEUIL_DETECTION is not None else []),
    f"- Borne (indices choisis à l'avance : {', '.join(indices.columns)}) : "
    + (f"lien démontré pour {', '.join(LIENS_DEMONTRES)} ; " if LIENS_DEMONTRES else "aucun lien démontré ; ")
    + f"aucun n'explique plus de {BORNE:.0%} de la variance du QIT ({BORNE_AJUSTEE:.0%} au-delà des covariables "
    f"cliniques), intervalles simultanés à 95 % ; un lien de {VARIANCE_DETECTABLE:.0%} aurait été détecté 8 fois sur 10"
    + (f". Le lien brut de {', '.join(BRUT_SEULEMENT)} disparaît après ajustement sur les covariables cliniques."
       if BRUT_SEULEMENT else "."),
    f"- Intervalle de prédiction à {1 - ALPHA:.0%} (prédiction conforme) : ± {q_final:.1f} points, contre "
    f"± {conforme.q_ref.mean():.1f} autour du QIT moyen.",
    "",
    "## Contrôle positif, classification, hypothèses",
    (f"- Contrôle positif : HRV seule → prématurité, AUC {controle['AUC']:.2f} "
     f"{outils.ic(controle['AUC IC bas'], controle['AUC IC haut'])}." if controle is not None else
     "- Contrôle positif : prématurité indisponible ou trop rare."),
    f"- Classification QIT < {SEUIL_QIT} ({bas.sum()} enfants) : meilleure AUC {classif['AUC']:.2f} "
    f"{outils.ic(classif['AUC IC bas'], classif['AUC IC haut'])} ({nom_classif}), exactitude équilibrée "
    f"{classif['exactitude équilibrée']:.2f}, kappa {classif['kappa']:.2f}.",
    *[f"- {a} ({int(h.enfants)} enfants) : gain {h.gain:+.2f} {outils.ic(h['IC bas'], h['IC haut'])} "
      f"({'soutenue' if h['IC bas'] > 0 else 'non soutenue'})." for a, h in hypotheses.iterrows()],
    f"- Avec {len(hypotheses)} hypothèses testées, une hypothèse soutenue isolée peut être due au hasard : à confirmer.",
    "",
    "## Limites",
    "- Environ 190 enfants ; pas de cohorte externe ; le QI à 3 ans dépend de nombreux facteurs non mesurés ici "
    "(environnement familial, niveau socio-économique).",
    "- Une association n'est pas une cause : un modèle prédictif ne dit pas comment améliorer le QI d'un enfant.",
    "",
    "## Suite",
    "- Variables tirées des données brutes (fichiers RR, signal ECG) : travail du groupe B.",
    "- Confirmer sur une autre cohorte toute hypothèse soutenue ci-dessus.",
])
(DOSSIER_RESULTATS / "SUMMARY.md").write_text(SUMMARY + "\n", encoding="utf-8")
display(Markdown(SUMMARY))

# 11. Avant de committer

Le plan demande de vérifier qu'aucune donnée de santé n'est suivie par git. `results/` ne contient que des chiffres agrégés ; `data/` reste local.

In [ ]:
suivis = subprocess.run(["git", "ls-files"], cwd=RACINE, capture_output=True, text=True).stdout.splitlines()
suspects = [f for f in suivis if re.search(r"\.(csv|xlsx?|mat|edf|zip|pkl|joblib)$", f, re.I)]
print("Fichiers de données suivis par git :", suspects or "aucun")
print("Les sorties de ce notebook sont vidées au commit par nbstripout ; sans ce filtre, faire « Restart & Clear Output ».")